In [1]:
import torch.nn as nn
from girth import grm_mml
from torch.utils.data import dataset


In [2]:
import torch
from torch import nn
from torch.nn import functional as F


class VSOSPred(nn.Module):
    def __init__(self, num_usrs: int, num_tasks: int):
        super().__init__()
        self.global_strong = nn.Embedding(num_usrs, 1)
        self.performance = nn.Embedding(num_usrs * num_tasks, 1)
        nn.init.zeros_(self.global_strong.weight)
        nn.init.zeros_(self.performance.weight)
        self.num_tasks = num_tasks
    def get_strong(self, usr_ids, task_ids):
        global_strength = self.global_strong(usr_ids).squeeze(-1)
        joint_ids = usr_ids * self.num_tasks + task_ids
        local = self.performance(joint_ids).squeeze(-1)
        return global_strength + local
    def forward(self, usr_a_ids, usr_b_ids, task_ids):
        strength_a = self.get_strong(usr_a_ids, task_ids)
        strength_b = self.get_strong(usr_b_ids, task_ids)
        return {"logits": strength_a - strength_b, "strength_a": strength_a, "strength_b": strength_b}
    def regularization_loss(self, global_weight, offset_weight):
        global_penalty = self.global_strong.weight.pow(2).mean()
        offset_penalty = self.performance.weight.pow(2).mean()
        return (global_weight * global_penalty + offset_weight * offset_penalty)

In [1]:
from torch.utils.data import Dataset
import numpy as np
import pandas as pd
from pathlib import Path


class VSOSet(Dataset):
    def __init__(self, df):
        self.cols = df.columns[1:].tolist()
        for col in self.cols:
            df[col].isnull().sum()


    '''def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]'''



In [4]:
import pandas as pd
df = pd.read_csv('learn_data.csv')
df.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
name = df.name.values
df.drop(columns=['name'], inplace=True)
#df = df.drop(columns=[''])
df

,id,total1,result1,result2,total3,grade2,stage,A,B,C,...,D1_y,A2_y,B2_y,C2_y,D2_y,Sum,place,id_y,title,grade
0,0,NaN,NaN,NaN,286.0,10.0,2.0,100.0,43.0,0.0,...,18.0,100.0,65.5,38.0,25.0,388.5,NaN,NaN,NaN,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,47.0,30.0,569.0,NaN,NaN,NaN,NaN
2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,100.0,100.0,68.0,10.0,531.0,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18.0,100.0,100.0,50.0,NaN,568.0,NaN,NaN,NaN,NaN
4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,100.0,100.0,39.0,10.0,521.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,100.0,100.0,61.0,NaN,561.0,NaN,NaN,NaN,NaN
471,471,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,100.0,100.0,39.0,NaN,486.0,NaN,NaN,NaN,NaN
472,472,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,0.0,NaN,NaN,230.0,NaN,NaN,NaN,NaN
473,473,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.0,100.0,47.0,100.0,719.0,NaN,NaN,NaN,NaN


In [3]:
num_usrs = 500

In [ ]:
model = VSOSpred(
    num_usrs=num_usrs,
    num_olimps=7,
)
optim = torch.optim.AdamW(model.parameters())
for batch in dataloader:
    out = model(
        athlete_a_ids=batch["athlete_a"],
        athlete_b_ids=batch["athlete_b"],
        competition_ids=batch["competition_id"],
    )
    pairwise_loss = F.binary_cross_entropy_with_logits(
        out["logits"],
        batch["target"].float(),
    )
    loss = (
        pairwise_loss
        + model.regularization_loss(1e-3, 1e-1)
    )
    loss.backward()
    optim.step()
    optim.zero_grad()